# Stages 4–5 — Feature Collapse, Targets & Decay Panel

**Stage 4:** Aggregate intraday substrate → one feature row per auction (`df_forecast`).  
All output columns known by the auction-day close — no lookahead.

**Stage 5:**
- Add h-day-forward curve-change targets (`d_level_h1`, `d_slope_h1`, `d_curv_h1`, `d_30y_h1`).
- Build long `decay_panel` (auction × horizon h) for local-projection IRF (Stage 7a).

**Reads:**
- `data/cache/intraday_stage1.parquet`
- `data/cache/bloomberg.parquet`
- `data/cache/regime_stage3.parquet`
- `data/cache/daily_stage2.parquet`

**Writes:**
- `data/cache/df_forecast.parquet`
- `data/cache/decay_panel.parquet`

Ref: Litterman-Scheinkman (1991) — level/slope/curvature decomposition.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import CACHE_DIR, MATURITIES, N_PCA, H_FORECAST, H_DECAY, TARGET
from src.features import (
    collapse_to_auction,
    fit_daily_pca,
    build_targets,
    build_decay_panel,
)

intraday_df  = pd.read_parquet(CACHE_DIR / 'intraday_stage1.parquet')
bloomberg_df = pd.read_parquet(CACHE_DIR / 'bloomberg.parquet')
regime_df    = pd.read_parquet(CACHE_DIR / 'regime_stage3.parquet')
daily_df     = pd.read_parquet(CACHE_DIR / 'daily_stage2.parquet')

print(f'Intraday : {len(intraday_df):,} rows')
print(f'Bloomberg: {len(bloomberg_df)} auctions')
print(f'Regime   : {len(regime_df)} auctions')
print(f'Daily    : {len(daily_df)} rows')

## Stage 4 — Collapse to auction level

In [ ]:
df_forecast = collapse_to_auction(intraday_df, bloomberg_df, regime_df)
print(f'df_forecast: {len(df_forecast)} auctions × {len(df_forecast.columns)} features')
df_forecast.head()

In [ ]:
# NA rate per feature
na_rate = df_forecast.isna().mean().sort_values(ascending=False)
print('Top NA columns:')
print(na_rate[na_rate > 0].head(15))

## Stage 5a — Daily PCA basis + targets

> Full-sample PCA for inspection. Inside CV folds, `fit_daily_pca` is called train-only.

In [ ]:
train_mask_all = np.ones(len(daily_df), dtype=bool)
pca_daily, pc_scores = fit_daily_pca(daily_df, train_mask_all)
daily_with_pcs = daily_df.join(pc_scores)

df_forecast = build_targets(df_forecast, daily_with_pcs, h=H_FORECAST)
target_cols = [c for c in df_forecast.columns if c.startswith('d_')]
print('Target columns:', target_cols)
df_forecast[target_cols].describe()

In [ ]:
fig, axes = plt.subplots(1, len(target_cols), figsize=(14, 3))
for ax, col in zip(axes, target_cols):
    df_forecast[col].hist(bins=30, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Stage 5b — Decay panel (auction × horizon)

In [ ]:
decay_panel = build_decay_panel(df_forecast, daily_with_pcs, target_col='PC1', H=H_DECAY)
print(f'Decay panel: {len(decay_panel):,} rows  |  horizons: {sorted(decay_panel["h"].unique())}')
decay_panel.head()

In [ ]:
# Mean cum_change by horizon (unconditional IRF shape preview)
mean_by_h = decay_panel.groupby('h')['cum_change'].mean()
mean_by_h.plot(marker='o', figsize=(8, 4))
plt.axhline(0, color='k', linewidth=0.8)
plt.xlabel('Horizon (business days)')
plt.ylabel('Mean cum_change')
plt.title('Unconditional cumulative curve change by horizon')
plt.show()

## Persist

In [ ]:
df_forecast.to_parquet(CACHE_DIR / 'df_forecast.parquet', index=False)
decay_panel.to_parquet(CACHE_DIR / 'decay_panel.parquet', index=False)
print('Saved → cache/df_forecast.parquet')
print('Saved → cache/decay_panel.parquet')